In [1]:
import os
from abc import ABC, abstractmethod
from enum import Enum
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

In [2]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

In [3]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [4]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [5]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [6]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

In [7]:
system_message = SystemMessage("You're an AI assistant.")
user_message = UserMessage("Hello, please introduce yourself!")

In [8]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

In [9]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")
        return AssistantMessage(response.choices[0].message.content or "")

In [10]:
provider = OpenAIProvider()
message = provider.call(system_message, [user_message])
print(message.summary())

Hello! I’m an AI assistant here to help with answering questions, brainstorming, writing, coding, explaining concepts, and more.

A few things I can do:
- Answer questions on a wide range of topics
- Help draft emails, messages, essays, and reports
- Assist with coding and debugging
- Summarize or explain complex information
- Brainstorm ideas and outline plans

If you’d like, you can tell me what you need help with, and I’ll jump in.


In [11]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-5-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")
        return AssistantMessage("".join(block.text for block in response.content if block.type == "text"))

In [12]:
provider = ClaudeProvider()
message = provider.call(system_message, [user_message])
print(message.summary())

Hello! I'm an AI assistant, here to help with a wide range of tasks. I can answer questions, explain concepts, help with writing and editing, brainstorm ideas, work through math or coding problems, summarize information, and much more. I don't have personal experiences or memories between conversations, but I'm glad to help with whatever you're working on. What can I do for you today?
